In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install fastnode2vec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 50.9 MB/s eta 0:00:00


# Lib


In [3]:
import os
import gc
import time
import json
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from fastnode2vec import Graph, Node2Vec

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, roc_curve, confusion_matrix, classification_report
from sklearn.calibration import CalibratedClassifierCV

import warnings
warnings.filterwarnings('ignore')

# Config

In [4]:
config = {
    # Feature extraction
    'use_node_features': True,
    'use_edge_heuristics': True,
    'use_skill_similarity': True,

    # Node Embedding
    'use_node2vec': True,

    # Column names
    'source_col': 'source',
    'target_col': 'target',
    'label_col': 'label',

    # Path
    "train_path": "/content/drive/MyDrive/colab/Social/train.csv",
    "test_path": "/content/drive/MyDrive/colab/Social/test.csv",
    "feature_json_path": "/content/drive/MyDrive/colab/Social/musae_git_features.json"
}

# Func

## Graph

In [5]:
def build_graph(train_df, test_df, source_col='source', target_col='target', label_col='label'):
    G = nx.Graph()
    train_nodes = set(train_df[source_col].unique()) | set(train_df[target_col].unique())
    test_nodes = set(test_df[source_col].unique()) | set(test_df[target_col].unique())
    all_nodes = train_nodes | test_nodes
    G.add_nodes_from(all_nodes)
    positive_edges = train_df[train_df[label_col] == 1][[source_col, target_col]].values
    G.add_edges_from(positive_edges)
    print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    return G

## Feature extraction

In [6]:
def extract_node_level_features(G, df, source_col='source', target_col='target', use=True):
    if not use:
        return pd.DataFrame(index=df.index)
    deg_cent = nx.degree_centrality(G)
    try:
        eig_cent = nx.eigenvector_centrality(G, max_iter=500, tol=1e-5)
    except nx.PowerIterationFailedConvergence:
        eig_cent = nx.eigenvector_centrality_numpy(G)
    pagerank = nx.pagerank(G, alpha=0.85, tol=1e-5)
    features = pd.DataFrame(index=df.index)
    centralities = [('deg', deg_cent), ('eig', eig_cent), ('pr', pagerank)]
    for name, cent_dict in centralities:
        features[f'{name}_source'] = df[source_col].map(cent_dict)
        features[f'{name}_target'] = df[target_col].map(cent_dict)
    print("[Extract] Completed extract node-level features")
    return features.fillna(0)

In [7]:
def extract_edge_level_features(G, df, source_col='source', target_col='target', use=True):
    if not use:
        return pd.DataFrame(index=df.index)
    features = pd.DataFrame(index=df.index)
    ebunch = list(zip(df[source_col], df[target_col]))
    features['common_neighbors'] = [len(list(nx.common_neighbors(G, u, v))) for u, v in ebunch]
    jac_gen = nx.jaccard_coefficient(G, ebunch)
    features['jaccard'] = [val for _, _, val in jac_gen]
    aa_gen = nx.adamic_adar_index(G, ebunch)
    features['adamic_adar'] = [val for _, _, val in aa_gen]
    def sp_dist(u, v):
        try:
            return nx.shortest_path_length(G, u, v)
        except nx.NetworkXNoPath:
            return len(G.nodes()) + 1
    features['shortest_path'] = [sp_dist(u, v) for u, v in ebunch]
    print("[Extract] Completed extract edge-level features")
    return features

In [8]:
def extract_skill_similarity(df, json_path, source_col='source', target_col='target', use=True, feature_prefix='skill'):
    if not use:
        return pd.DataFrame(index=df.index)
    with open(json_path, 'r', encoding='utf-8') as f:
        raw_dict = json.load(f)
    str_entity_dict = {str(k): set(v) for k, v in raw_dict.items()}
    su_list = [str_entity_dict.get(str(x), set()) for x in df[source_col]]
    sv_list = [str_entity_dict.get(str(x), set()) for x in df[target_col]]
    inter_len = np.array([len(u & v) for u, v in zip(su_list, sv_list)])
    len_u = np.array([len(u) for u in su_list])
    len_v = np.array([len(v) for v in sv_list])
    union_len = len_u + len_v - inter_len
    features = pd.DataFrame(index=df.index)
    col_common = f'{feature_prefix}_common'
    col_jaccard = f'{feature_prefix}_jaccard'
    col_dice = f'{feature_prefix}_dice'
    col_cosine = f'{feature_prefix}_cosine'
    col_overlap = f'{feature_prefix}_overlap_ratio'
    features[col_common] = inter_len
    with np.errstate(divide='ignore', invalid='ignore'):
        features[col_jaccard] = np.where(union_len > 0, inter_len / union_len, 0.0)
        features[col_dice] = np.where((len_u + len_v) > 0, (2.0 * inter_len) / (len_u + len_v), 0.0)
        features[col_cosine] = np.where((len_u * len_v) > 0, inter_len / np.sqrt(len_u * len_v), 0.0)
        min_len = np.minimum(len_u, len_v)
        features[col_overlap] = np.where(min_len > 0, inter_len / min_len, 0.0)
    print(f"[Extract] Completed extract skill similarity")
    return features

## Graph embedding

In [9]:
def get_node2vec_embeddings(G):
    is_weighted = any('weight' in edge_data for _, _, edge_data in G.edges(data=True))
    if is_weighted:
        edges = [(u, v, data.get('weight', 1.0)) for u, v, data in G.edges(data=True)]
    else:
        edges = list(G.edges())
    n2v_graph = Graph(
        edges,
        directed=G.is_directed(),
        weighted=is_weighted,
        number_of_edges=len(edges)
    )
    model = Node2Vec(
        n2v_graph,
        dim=128,
        walk_length=80,
        window=10,
        p=0.5,
        q=2.0,
        workers=4,
        batch_walks=10000
    )
    model.train(epochs=10)
    embeddings = {}
    for node in G.nodes():
        node_str = str(node)
        if node_str in model.wv:
            embeddings[node] = model.wv[node_str]
        else:
            embeddings[node] = np.zeros(32, dtype=np.float32)
    print("[Embedding] Completed node embeddings")
    return embeddings

In [10]:
def get_edge_embeddings(df, embeddings, source_col='source', target_col='target'):
    if embeddings is None:
        return pd.DataFrame(index=df.index)
    dim = len(next(iter(embeddings.values())))
    hadamard_list = []
    for u, v in zip(df[source_col], df[target_col]):
        emb_u = embeddings.get(u, np.zeros(dim))
        emb_v = embeddings.get(v, np.zeros(dim))
        hadamard_list.append(emb_u * emb_v)
    columns = [f'n2v_had_{i}' for i in range(dim)]
    print("[Embedding] Completed edge embeddings")
    return pd.DataFrame(hadamard_list, columns=columns, index=df.index)

## Model

In [11]:
def train_model(model, X_train, y_train):
    model.fit(X_train, y_train)
    print(f"[Train] Completed")
    return model

## Eval

In [12]:
def precision_at_k(y_true, y_score, k):
    idx = np.argsort(y_score)[::-1][:k]
    y_topk = y_true[idx]
    return np.sum(y_topk) / k

def recall_at_k(y_true, y_score, k):
    idx = np.argsort(y_score)[::-1][:k]
    y_topk = y_true[idx]
    total_positive = np.sum(y_true)
    return np.sum(y_topk) / total_positive if total_positive > 0 else 0

def hits_at_k(y_true, y_score, k):
    idx = np.argsort(y_score)[::-1][:k]
    y_topk = y_true[idx]
    return 1 if np.sum(y_topk) > 0 else 0

def mrr_score(y_true, y_score):
    idx = np.argsort(y_score)[::-1]
    y_true_sorted = np.array(y_true)[idx]
    positive_indices = np.where(y_true_sorted == 1)[0]
    if len(positive_indices) > 0:
        first_positive_rank = positive_indices[0] + 1
        return 1.0 / first_positive_rank
    else:
        return 0.0

def evaluate_global(y_true, y_prob):
    metrics = {
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "AUPR": average_precision_score(y_true, y_prob),
        "MRR": mrr_score(y_true, y_prob)
    }

    df = pd.DataFrame(metrics.items(), columns=["Metric", "Value"])
    return df

def evaluate_at_k(y_true, y_prob, k_list=[10, 100, 1000]):
    k_metrics = []
    for k in k_list:
        k_metrics.append({
            "k": k,
            "Precision@K": precision_at_k(y_true, y_prob, k),
            "Recall@K": recall_at_k(y_true, y_prob, k),
            "Hits@K": hits_at_k(y_true, y_prob, k)
        })

    df = pd.DataFrame(k_metrics)

    return df

In [13]:
def visualize_confusion_matrix(y_true, y_probs, threshold=0.5):
    y_pred = (y_probs >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    sns.kdeplot(y_probs[y_true == 0], fill=True, color="crimson", label="Actual: Label 0", ax=axes[0], alpha=0.5)
    sns.kdeplot(y_probs[y_true == 1], fill=True, color="teal", label="Actual: Label 1", ax=axes[0], alpha=0.5)
    axes[0].axvline(x=threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold = {threshold}')
    axes[0].set_title('Predicted Probability Distribution', fontsize=14)
    axes[0].set_xlabel('Predicted Probability of Label 1', fontsize=12)
    axes[0].set_ylabel('Density', fontsize=12)
    axes[0].set_xlim([0, 1])
    axes[0].legend()

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, annot_kws={"size": 14}, ax=axes[1])
    axes[1].set_title(f'Confusion Matrix (Threshold = {threshold})', fontsize=14)
    axes[1].set_xlabel('Predicted', fontsize=12)
    axes[1].set_ylabel('Actual', fontsize=12)
    axes[1].set_xticklabels(['Label 0', 'Label 1'])
    axes[1].set_yticklabels(['Label 0', 'Label 1'])

    plt.tight_layout(pad=0.5)
    plt.show()

# Main

## Load data

In [14]:
train_df = pd.read_csv(config['train_path'])
test_df = pd.read_csv(config['test_path'])

## Build graph

In [15]:
G = build_graph(train_df, test_df, source_col=config['source_col'], target_col=config['target_col'], label_col=config['label_col'])

Graph: 37700 nodes, 231203 edges


## Feature extraction

In [16]:
node_train = extract_node_level_features(G, train_df, config['source_col'], config['target_col'], config['use_node_features'])
edge_train = extract_edge_level_features(G, train_df, config['source_col'], config['target_col'], config['use_edge_heuristics'])
skill_train = extract_skill_similarity(train_df, config['feature_json_path'], config['source_col'], config['target_col'], config['use_skill_similarity'])

[Extract] Completed extract node-level features
[Extract] Completed extract edge-level features
[Extract] Completed extract skill similarity


In [17]:
node_test  = extract_node_level_features(G, test_df,  config['source_col'], config['target_col'], config['use_node_features'])
edge_test  = extract_edge_level_features(G, test_df,  config['source_col'], config['target_col'], config['use_edge_heuristics'])
skill_test  = extract_skill_similarity(test_df,  config['feature_json_path'], config['source_col'], config['target_col'], config['use_skill_similarity'])

[Extract] Completed extract node-level features
[Extract] Completed extract edge-level features
[Extract] Completed extract skill similarity


## Node embeddings

In [18]:
embeddings = None
if config['use_node2vec']:
    embeddings = get_node2vec_embeddings(G)

Reading graph:   0%|          | 0/231203 [00:00<?, ?it/s]

Epochs:   0%|          | 0/10 [00:00<?, ?it/s]

[Embedding] Completed node embeddings


## Edge embeddings

In [19]:
n2v_train = get_edge_embeddings(train_df, embeddings, config['source_col'], config['target_col'])
n2v_test  = get_edge_embeddings(test_df,  embeddings, config['source_col'], config['target_col'])

[Embedding] Completed edge embeddings
[Embedding] Completed edge embeddings


## Matrix features

In [20]:
def preprocessing(X_train, X_test):
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler

In [21]:
node_train, node_test, _ = preprocessing(node_train, node_test)
edge_train, edge_test, _  = preprocessing(edge_train, edge_test)
skill_train, skill_test, _ = preprocessing(skill_train, skill_test)

node_train = pd.DataFrame(node_train, index=train_df.index)
node_test = pd.DataFrame(node_test, index=test_df.index)
edge_train = pd.DataFrame(edge_train, index=train_df.index)
edge_test  = pd.DataFrame(edge_test,  index=test_df.index)
skill_train = pd.DataFrame(skill_train, index=train_df.index)
skill_test  = pd.DataFrame(skill_test,  index=test_df.index)

In [22]:
X_train = pd.concat([node_train, edge_train, skill_train, n2v_train], axis=1)
X_test  = pd.concat([node_test,  edge_test,  skill_test,  n2v_test],  axis=1)

In [23]:
X_train = X_train.values
X_test = X_test.values
y_train = train_df[config['label_col']]
y_test  = test_df[config['label_col']]

## Train

In [24]:
model = LogisticRegression(max_iter=1000)

In [25]:
lr_model = train_model(model, X_train, y_train)

[Train] Completed


## Eval

In [27]:
y_pred = lr_model.predict(X_test)
y_probs = lr_model.predict_proba(X_test)[:, 1]

In [28]:
evaluate_global(y_true=y_test, y_prob=y_probs).round(4)

,Metric,Value
0,ROC-AUC,0.9104
1,AUPR,0.9206
2,MRR,1.0000


In [29]:
evaluate_at_k(y_true=y_test, y_prob=y_probs, k_list=[10, 50, 100, 500, 1000, 10000]).round(4)

,k,Precision@K,Recall@K,Hits@K
0,10,1.0000,0.0002,1
1,50,1.0000,0.0009,1
2,100,1.0000,0.0017,1
3,500,1.0000,0.0087,1
4,1000,1.0000,0.0173,1
5,10000,0.9939,0.1720,1


## Export

In [30]:
model_save_path = '/content/drive/MyDrive/colab/Social/logistic.joblib'
joblib.dump(model, model_save_path)
print(f"[Export] Model exported successfully to {model_save_path}")

[Export] Model exported successfully to /content/drive/MyDrive/colab/Social/logistic.joblib
